# Crack segmentation: train the U-Net on a GPU (Google Colab)

Runtime → Change runtime type → **T4 GPU**, then run all cells. Training takes about 10–15 minutes.

The DeepCrack images (Liu et al., Neurocomputing 2019) are downloaded from the authors' repository; they may only be used for non-commercial research and education.

In [ ]:
!git clone https://github.com/elmoussaouioumayma/deep-learning-crack-segmentation.git
%cd deep-learning-crack-segmentation
!pip -q install opencv-python-headless scikit-image

In [ ]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU - switch the runtime')

## 1. Classical baseline (tuned on training images)

In [ ]:
!python tune_classical.py

## 2. Train the U-Net
The best epoch and the decision threshold are chosen on 40 held-out training images.

In [ ]:
!python train.py --epochs 80 --base 32 --batch 16 --repeats 4

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('results/training_log.csv')
ax = log.plot(x='epoch', y=['train_loss', 'val_f1'], secondary_y='val_f1', figsize=(7, 3)); plt.show()

## 3. Evaluate on the 237 test images

In [ ]:
!mkdir -p weights && cp results/unet.pt weights/unet.pt
!python evaluate.py

In [ ]:
from IPython.display import Image, display
for f in ['metrics.png', 'per_image_f1.png', 'examples.png', 'failure_cases.png']:
    display(Image('images/' + f))

## 4. Download the trained weights

In [ ]:
from google.colab import files
files.download('weights/unet.pt')